In [1]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
import re
import numpy as np

from sklearn.preprocessing import OneHotEncoder

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

In [3]:
# 東京都23区のリスト
wards = [
    "千代田区", "中央区", "港区", "新宿区", "文京区", "台東区", "墨田区", "江東区",
    "品川区", "目黒区", "大田区", "世田谷区", "渋谷区", "中野区", "杉並区", "豊島区",
    "北区", "荒川区", "板橋区", "練馬区", "足立区", "葛飾区", "江戸川区"
]

In [4]:
# 正規表現を使って「何区」を抽出する関数
def extract_ward(address):
    # 23区リストを元に住所から該当する区を検索
    for ward in wards:
        if ward in address:
            return ward
    return None  # 該当する区がなければ None を返す

In [5]:
# 住所データを含む列に対して処理を適用
train['ward'] = train['所在地'].apply(extract_ward)
test['ward'] = test['所在地'].apply(extract_ward)

In [6]:
# wardのOne-Hot Encoding
train = pd.get_dummies(train, columns=['ward'])
test = pd.get_dummies(test, columns=['ward'])

In [7]:
# 徒歩時間を正規表現で抽出し、最短のものを取得する関数
def extract_min_walk_time(access_str):
    # 正規表現で「徒歩○分」を抽出
    walk_times = re.findall(r'徒歩(\d+)分', access_str)
    # 数字に変換して最小値を返す
    if walk_times:
        return min(map(int, walk_times))
    return None

In [8]:
# 最短時間の列を追加
train['min_walk'] = train['アクセス'].apply(extract_min_walk_time)
test['min_walk'] = test['アクセス'].apply(extract_min_walk_time)

In [9]:
# 部屋数を数値に変換する関数
def extract_room_count(madori):
    match = re.search(r'\d+', madori)  # 部屋数（数字）を抽出
    if match:
        return int(match.group())
    return 0  # 部屋数がない場合は0にする

In [10]:
# 部屋タイプを抽出する関数
def extract_room_type(madori):
    types = {
        'R': 0, 'L': 0, 'D': 0, 'K': 0, 'S': 0
    }
    for t in types.keys():
        if t in madori and t not in ['DK', 'LDK']:  # 「DK」「LDK」を除く
            types[t] = 1
    return types

In [11]:
# 部屋数を新しい列に追加
train['部屋数'] = train['間取り'].apply(extract_room_count)
test['部屋数'] = test['間取り'].apply(extract_room_count)

In [12]:
# 部屋タイプを新しい特徴量に追加
room_types = train['間取り'].apply(extract_room_type)
train = pd.concat([train, room_types.apply(pd.Series)], axis=1)
room_types2 = test['間取り'].apply(extract_room_type)
test = pd.concat([test, room_types2.apply(pd.Series)], axis=1)

In [13]:
# 築年数を数値に変換する関数
def convert_age(age_str):
    # 正規表現で「年」と「ヶ月」を分離
    match = re.match(r'(\d+)年(\d+)ヶ月', age_str)
    if match:
        years = int(match.group(1))  # 年
        months = int(match.group(2))  # ヶ月
        # 年とヶ月を数値に変換 (1年 = 12ヶ月)
        return years + months / 12
    else:
        # ○年だけの場合
        match = re.match(r'(\d+)年', age_str)
        if match:
            years = int(match.group(1))
            return years
        # ○ヶ月だけの場合
        match = re.match(r'(\d+)ヶ月', age_str)
        if match:
            months = int(match.group(1))
            return months / 12
        return None  # 不正なデータはNoneとする

In [14]:
# 築年数列を変換
train['築年数_変換'] = train['築年数'].apply(convert_age)
test['築年数_変換'] = test['築年数'].apply(convert_age)

In [15]:
# NaNを0に置き換え
train['築年数_変換'] = train['築年数_変換'].fillna(0).astype(int)
test['築年数_変換'] = test['築年数_変換'].fillna(0).astype(int)

In [16]:
# 正規表現で数値部分のみを抽出
train['面積_数値'] = test['面積'].str.extract(r'(\d+\.\d+|\d+)').astype(float)
test['面積_数値'] = test['面積'].str.extract(r'(\d+\.\d+|\d+)').astype(float)

In [17]:
# 所在階列の「階」と「階建」を取り出して計算する関数
def calc_floor_ratio(floor_info):
    try:
        # 「階」と「階建」を分割
        floor, total_floors = floor_info.split('階／')
        # 数値に変換して割り算
        return int(floor) / int(total_floors.replace('階建', ''))
    except:
        return None  # エラー時はNoneを返す

In [18]:
# 新しい列を作成
train['所在階_数値'] = train['所在階'].apply(calc_floor_ratio)
test['所在階_数値'] = test['所在階'].apply(calc_floor_ratio)

In [19]:
# 欠損値の補完（平均値で補完）
mean_value = train['所在階_数値'].mean()  # 平均値を計算
train['所在階_数値'] = train['所在階_数値'].fillna(mean_value)  # 平均値で欠損値を補完
mean_value2 = test['所在階_数値'].mean()  # 平均値を計算
test['所在階_数値'] = test['所在階_数値'].fillna(mean_value2)  # 平均値で欠損値を補完

In [20]:
# 建物構造のOne-Hot Encoding
train = pd.get_dummies(train, columns=['建物構造'])
test = pd.get_dummies(test, columns=['建物構造'])

In [21]:
# 不要な列を削除
train.drop(['所在地', 'アクセス', '間取り', '築年数', '方角', '面積', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間'], axis=1, inplace=True)
test.drop(['所在地', 'アクセス', '間取り', '築年数', '方角', '面積', '所在階', 'バス・トイレ', 'キッチン', '放送・通信', '室内設備', '駐車場', '周辺環境', '契約期間'], axis=1, inplace=True)

In [22]:
plt.rcParams['font.family'] = 'IPAexGothic'

In [23]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31470 entries, 0 to 31469
Data columns (total 45 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                31470 non-null  int64  
 1   賃料                                31470 non-null  int64  
 2   ward_世田谷区                         31470 non-null  bool   
 3   ward_中央区                          31470 non-null  bool   
 4   ward_中野区                          31470 non-null  bool   
 5   ward_北区                           31470 non-null  bool   
 6   ward_千代田区                         31470 non-null  bool   
 7   ward_台東区                          31470 non-null  bool   
 8   ward_品川区                          31470 non-null  bool   
 9   ward_墨田区                          31470 non-null  bool   
 10  ward_大田区                          31470 non-null  bool   
 11  ward_文京区                          31470 non-null  bool   
 12  ward

In [24]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31262 entries, 0 to 31261
Data columns (total 45 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   id                                31262 non-null  int64  
 1   ward_世田谷区                         31262 non-null  bool   
 2   ward_中央区                          31262 non-null  bool   
 3   ward_中野区                          31262 non-null  bool   
 4   ward_北区                           31262 non-null  bool   
 5   ward_千代田区                         31262 non-null  bool   
 6   ward_台東区                          31262 non-null  bool   
 7   ward_品川区                          31262 non-null  bool   
 8   ward_墨田区                          31262 non-null  bool   
 9   ward_大田区                          31262 non-null  bool   
 10  ward_文京区                          31262 non-null  bool   
 11  ward_新宿区                          31262 non-null  bool   
 12  ward

In [25]:
plt.rcParams['font.family'] = 'IPAexGothic'

In [26]:
sns.pairplot(train)

In [27]:
train.to_csv('train_pre.csv', index=False)
test.to_csv('test_pre.csv', index=False)